# **基于静态Tensor的CV融合算子高级开发**

## 概述

本节使用 Ascend C 基础 API 实现 `MMAD + GELU` CV 融合算子。M/N 二维核心映射将输出矩阵分配给 32 个 AI Core；Cube Core 执行分块 MMAD，两个 Vector Core 通过 L0C-UB 路径接收结果并执行 RegBase GELU。精度与性能数据在 3510 上采集。

### 学习前置要求

学习本小节前，建议已经具备以下基础：

- 掌握基于静态 Tensor 的 CV 融合算子基础开发流程；
- 掌握静态 Tensor 地址与容量管理、Cube 数据路径和核内同步；

### 学习目标

完成本小节后，开发者应能够：

1. 根据算子 Shape 设计 M/N 核心映射和片上 Tile；
2. 使用静态 Tensor 组织 Cube Core 的多级搬运与计算流水；
3. 使用 RegBase 编写连续依赖的 Vector 算术链；
4. 完成 Cube Core 与两个 Vector Core 之间的数据交接；
5. 使用统一口径验证融合算子的精度与性能。

### 优化结构

本节从静态分块开始，依次完成 Cube Core 多级流水、Vector Core RegBase 计算和 CV 数据交接，最后通过精度与性能数据验证整体实现。

![mmad_gelu_adv 实现结构](images/07_02_static_tensor_cv_fusion_advanced/0_optimization_route.svg)

### 本节内容

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr><th align="left">章节</th><th align="left">内容</th><th align="left">学习期望</th></tr>
  </thead>
  <tbody>
    <tr><td>环境准备</td><td>初始化 CANN 环境</td><td>初始化环境正常</td></tr>
    <tr><td>算子计算分析</td><td>分析 MMAD-GELU 的输入输出、Shape 和计算语义</td><td>理解待实现算子的规格</td></tr>
    <tr><td>数据切分与算子框架</td><td>定义算子实现框架、核心映射和静态 Tensor</td><td>完成算子基本结构</td></tr>
    <tr><td>Cube Core 多级流水</td><td>实现批量搬运、Double Buffer 和硬件事件</td><td>完成 Cube Core 计算路径</td></tr>
    <tr><td>Vector Core RegBase</td><td>实现 GELU 寄存器计算</td><td>完成 Vector Core 计算路径</td></tr>
    <tr><td>CV 数据交接</td><td>实现 Fixpipe 双目标和跨核同步</td><td>完成融合算子实现</td></tr>
    <tr><td>编译、精度与性能</td><td>运行算子并分析同 Shape 对照数据</td><td>验证实现正确并理解优化效果</td></tr>
    <tr><td>课后实践</td><td>完成 GELU 寄存器双路展开</td><td>独立完成 RegBase 变体</td></tr>
  </tbody>
</table>

### 运行环境与硬件说明

- 本节目标架构为 `dav-3510`，运行模式为 `npu`；
- 融合 Kernel 使用 `__mix__(1, 2)`，每个 AI Core 包含 1 个 Cube Core 和 2 个 Vector Core。


---
## 1. 环境准备

运行下方代码，将 `set_env.sh` 导出的 CANN 环境变量写入当前 Jupyter 进程。


In [ ]:
import os
import subprocess
from pathlib import Path

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

!mkdir -p Source
!cp -a src/07_02_static_tensor_cv_fusion_advanced Source/
print("Source/07_02_static_tensor_cv_fusion_advanced 初始化完毕。")

### 1.1 目录结构介绍

代码目录如下：

```
├── src/07_02_static_tensor_cv_fusion_advanced
│   ├── scripts
│   │   ├── gen_data.py                         // 生成输入数据和 golden 数据
│   │   └── verify_result.py                    // 校验算子输出与 golden 数据
│   ├── CMakeLists.txt                          // 编译工程文件，定义 MMAD-GELU 样例目标
│   ├── data_utils.h                            // Host 侧文件读入写出工具
│   ├── gelu_unroll_practice.h                  // RegBase GELU 实现及课后实践代码
│   ├── mmad_gelu_adv_operator.h                // MMAD-GELU CV 融合算子设备侧实现
│   ├── mmad_gelu_adv.asc                       // Host 侧数据准备、Kernel 启动和结果回收
│   └── run.sh                                  // 输入生成、编译、运行和精度校验脚本
└── answer/07_02_static_tensor_cv_fusion_advanced
    ├── scripts/gen_data.py                     // 课后实践数据生成脚本参考答案
    ├── gelu_unroll_practice.h                  // 双路 RegBase GELU 参考答案
    ├── mmad_gelu_adv_operator.h                // 完整设备侧实现参考答案
    └── mmad_gelu_adv.asc                       // 完整 Host 侧实现参考答案
```

`mmad_gelu_adv.asc` 包含 Host 侧数据准备、Kernel 启动和结果回收，`mmad_gelu_adv_operator.h` 包含设备侧实现。CMake 仅构建 `mmad_gelu_adv`。


---
## 2. 算子计算分析

MMAD-GELU 由 MMAD 和 GELU 激活组成。MMAD 在 Cube Core 上执行，GELU 在 Vector Core 上执行。以下规格同时约束 Host 侧数据大小、核心映射和片上分块。

### 2.1 计算定义

目标算子依次执行 MMAD 和 GELU：

```text
C = A x B + Bias
Y = GELU(C)
```

![MMAD-GELU 计算定义](images/07_02_static_tensor_cv_fusion_advanced/2_operator_formula.svg)

GELU 使用如下近似公式：

```text
GELU(x) = x / (1 + exp(-1.595769 x (1 + 0.044715 x^2)))
```

### 2.2 输入输出规格

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr><th align="left">参数</th><th align="left">Shape</th><th align="left">数据类型</th><th align="left">布局与语义</th></tr>
  </thead>
  <tbody>
    <tr><td><code>A</code></td><td><code>[1920, 2048]</code></td><td><code>half</code></td><td>ND，MMAD 左矩阵</td></tr>
    <tr><td><code>B</code></td><td><code>[2048, 2048]</code></td><td><code>half</code></td><td>转置布局，MMAD 右矩阵</td></tr>
    <tr><td><code>Bias</code></td><td><code>[2048]</code></td><td><code>half</code></td><td>沿 N 维广播</td></tr>
    <tr><td><code>Y</code></td><td><code>[1920, 2048]</code></td><td><code>float32</code></td><td>MMAD-GELU 融合输出</td></tr>
  </tbody>
</table>

---
## 3. 数据切分与算子框架

### 3.1 切分策略

![M/N 二维核心映射](images/07_02_static_tensor_cv_fusion_advanced/3_core_partition.svg)

`singleCoreM=480`、`singleCoreN=256`，因此 M 方向划分为 4 组，N 方向划分为 8 组，共启动 32 个 AI Core。Cube Core 的 `GetBlockIdx()` 直接对应 AI Core 编号；Vector Core 侧每两个 Block 对应同一个 AI Core，因此需要除以 2。

**分块参数**
<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">参数</th><th align="left">取值</th><th align="left">含义</th></tr></thead>
  <tbody>
    <tr><td><code>singleCoreM</code></td><td>480</td><td>单个 AI Core 负责的最大 M 范围</td></tr>
    <tr><td><code>singleCoreN</code></td><td>256</td><td>单个 AI Core 负责的最大 N 范围</td></tr>
    <tr><td><code>baseM</code></td><td>240</td><td>单个 MMAD Tile 的 M</td></tr>
    <tr><td><code>baseK</code></td><td>64</td><td>单个 MMAD Tile 的 K</td></tr>
    <tr><td><code>baseN</code></td><td>256</td><td>单个 MMAD Tile 的 N</td></tr>
    <tr><td><code>stepKa / stepKb</code></td><td>4 / 4</td><td>每次进入 L1 的 K Tile 数量</td></tr>
  </tbody>
</table>

### 3.2 算子框架实现
`mmad_gelu_adv.asc` 包含 Host 侧数据准备、Kernel 启动和结果回收。设备侧类与 Kernel 入口定义在 `mmad_gelu_adv_operator.h` 中。

当前实现 `KernelMmadGelu`类声明和待实现优化。算子类将切分参数以模板参数形式传入。

In [ ]:
%%writefile Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv_operator.h
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

/* !
 * \file mmad_gelu_adv_operator.h
 * \brief 基于静态 Tensor 的 MMAD-GELU 高性能 CV 融合样例。
 */

#ifndef MMAD_GELU_ADV_OPERATOR_H
#define MMAD_GELU_ADV_OPERATOR_H

#include "kernel_operator.h"

__aicore__ __inline__ constexpr uint32_t DivCeil(uint32_t a, uint32_t b) { return (a + b - 1) / b; }

// half type, cube block: [16, 16]
constexpr uint32_t CUBE_BLOCK = 16;
constexpr uint32_t L0_PINGPONG_BYTES = 32 * 1024;
constexpr uint32_t L1_PINGPONG_BYTES = 256 * 1024;
constexpr bool IS_B_TRANSPOSE = true;
constexpr float GELU_COEFF_A = 0.044715f;
constexpr float GELU_COEFF_B = -1.595769f;

#include "gelu_unroll_practice.h"

// Fixpipe配置
constexpr AscendC::FixpipeConfig CFG_ROW_MAJOR_UB = {AscendC::CO2Layout::ROW_MAJOR, true};

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
class KernelMmadGelu {
public:
    __aicore__ inline KernelMmadGelu() {}

    __aicore__ inline void Init(
        __gm__ uint8_t* xMatrix, __gm__ uint8_t* yMatrix, __gm__ uint8_t* bias, __gm__ uint8_t* zMatrix);
    __aicore__ inline void Process();

private:
    __aicore__ inline void InitComputeParams();
    __aicore__ inline void InitAicSyncFlags();
    __aicore__ inline void WaitAicSyncFlags();
    __aicore__ inline void ProcessLoopAic(
        AscendC::LocalTensor<half>& a1LocalPing, AscendC::LocalTensor<half>& a1LocalPong,
        AscendC::LocalTensor<half>& a2LocalPing, AscendC::LocalTensor<half>& a2LocalPong,
        AscendC::LocalTensor<half>& b1LocalPing, AscendC::LocalTensor<half>& b1LocalPong,
        AscendC::LocalTensor<half>& b2LocalPing, AscendC::LocalTensor<half>& b2LocalPong,
        AscendC::LocalTensor<half>& bias1Local, AscendC::LocalTensor<float>& bias2Local,
        AscendC::LocalTensor<float>& cLocal, uint32_t mBlockIdx, uint32_t nBlockIdx);
    __aicore__ inline void ProcessLoopAiv(uint32_t mBlockIdx, uint32_t nBlockIdx);
    __aicore__ inline void GeluRegBaseCompute(
        const AscendC::LocalTensor<float>& xLocal, const AscendC::LocalTensor<float>& yLocal, uint32_t n);
    __aicore__ inline void DataCopyInA(
        AscendC::LocalTensor<half> a1Local, uint32_t kChunkIdx, uint32_t mBlockIdx);
    __aicore__ inline void DataCopyInB(
        AscendC::LocalTensor<half> b1Local, uint32_t kChunkIdx, uint32_t nBlockIdx);
    __aicore__ inline void DataLoadA(
        AscendC::LocalTensor<half> a1Local, AscendC::LocalTensor<half> a2Local, uint32_t mBlockIdx,
        uint32_t kOffsetInChunkA);
    __aicore__ inline void DataLoadB(
        AscendC::LocalTensor<half> b1Local, AscendC::LocalTensor<half> b2Local, uint32_t nBlockIdx,
        uint32_t kOffsetInChunkB);
    __aicore__ inline void PrepareBias(
        AscendC::LocalTensor<half>& bias1Local, AscendC::LocalTensor<float>& bias2Local, uint32_t nBlockIdx);
    __aicore__ inline void Compute(
        AscendC::LocalTensor<float> cLocal, AscendC::LocalTensor<half> a2Local,
        AscendC::LocalTensor<half> b2Local, AscendC::LocalTensor<float> bias2Local, uint32_t kBlockIdx,
        uint32_t mBlockIdx, uint32_t nBlockIdx);
    __aicore__ inline void CopyOutAic(
        AscendC::LocalTensor<float> cLocal, uint32_t mBlockIdx, uint32_t nBlockIdx);
    __aicore__ inline void GeluFromUB(uint32_t mBlockIdx, uint32_t nBlockIdx);

    AscendC::GlobalTensor<half> aGM;
    AscendC::GlobalTensor<half> bGM;
    AscendC::GlobalTensor<half> biasGM;
    AscendC::GlobalTensor<float> cGM;
    AscendC::GlobalTensor<half> aGMOri;
    AscendC::GlobalTensor<half> bGMOri;
    AscendC::GlobalTensor<half> biasGMOri;
    AscendC::GlobalTensor<float> cGMOri;

    AscendC::LocalTensor<float> xUB;
    AscendC::LocalTensor<float> geluOutUB;

    uint32_t actualSingleCoreM, actualSingleCoreN;
    uint32_t mLoopCount, nLoopCount, kLoopCount;
    uint32_t baseMCount, baseNCount;
    uint32_t tailM, tailN;
    uint32_t tailMAlign, tailNAlign;
    uint8_t mte1DBFlag = 0;

    static constexpr uint32_t geluOutUBAddr = baseM / 2 * baseN * sizeof(float);
};



### 3.3 计算参数初始化
`InitComputeParams` 根据核心编号计算 M/N 分区、GM 偏移和尾块长度。


In [ ]:
%%writefile -a Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv_operator.h

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::
    InitComputeParams()
    {
        constexpr uint32_t mIter = DivCeil(M, singleCoreM);
        // __mix__(1,2) 模式下，每个 AI Core 由 1 个 Cube Core 和 2 个 Vector Core 组成。
        // GetBlockIdx() 在 Cube Core 和 Vector Core 分支分别连续编号。
        uint32_t aiCoreId;
        if ASCEND_IS_AIC {
            aiCoreId = AscendC::GetBlockIdx();
        } else {
            aiCoreId = AscendC::GetBlockIdx() / 2;
        }
        uint32_t mIterIdx = aiCoreId % mIter;
        uint32_t nIterIdx = aiCoreId / mIter;

        uint64_t gmOffsetA = mIterIdx * singleCoreM * K;
        uint64_t gmOffsetB = IS_B_TRANSPOSE ? nIterIdx * K * singleCoreN : nIterIdx * singleCoreN;
        uint64_t gmOffsetC = mIterIdx * singleCoreM * N + nIterIdx * singleCoreN;
        aGM = aGMOri[gmOffsetA];
        bGM = bGMOri[gmOffsetB];
        biasGM = biasGMOri[nIterIdx * singleCoreN];
        cGM = cGMOri[gmOffsetC];

        actualSingleCoreM = M - mIterIdx * singleCoreM;
        actualSingleCoreM = actualSingleCoreM < singleCoreM ? actualSingleCoreM : singleCoreM;
        actualSingleCoreN = N - nIterIdx * singleCoreN;
        actualSingleCoreN = actualSingleCoreN < singleCoreN ? actualSingleCoreN : singleCoreN;

        kLoopCount = DivCeil(singleCoreK, baseK);
        mLoopCount = DivCeil(actualSingleCoreM, baseM);
        nLoopCount = DivCeil(actualSingleCoreN, baseN);

        baseNCount = actualSingleCoreN / baseN;
        tailN = actualSingleCoreN % baseN;
        tailNAlign = DivCeil(tailN, CUBE_BLOCK) * CUBE_BLOCK;

        baseMCount = actualSingleCoreM / baseM;
        tailM = actualSingleCoreM % baseM;
        tailMAlign = DivCeil(tailM, CUBE_BLOCK) * CUBE_BLOCK;
    }


### 3.4 Tile 容量与静态 Tensor

![Tile 片上存储容量](images/07_02_static_tensor_cv_fusion_advanced/3_tile_capacity.svg)

静态 Tensor 的地址和长度在编译期确定。A1/B1 与 A2/B2 分别预留 Ping/Pong 两个槽位，L0C 保存当前 `baseM x baseN` 的 MMAD 结果，两个 Vector Core 分别使用一半 M 行对应的 UB。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">存储</th><th align="left">单槽元素</th><th align="left">双槽占用</th><th align="left">约束</th></tr></thead>
  <tbody>
    <tr><td>A1</td><td><code>240 x 64 x 4</code> half</td><td>240 KiB</td><td>小于 256 KiB 分区</td></tr>
    <tr><td>B1</td><td><code>64 x 256 x 4</code> half</td><td>256 KiB</td><td>等于 256 KiB 分区</td></tr>
    <tr><td>A2</td><td><code>240 x 64</code> half</td><td>60 KiB</td><td>每槽小于 32 KiB</td></tr>
    <tr><td>B2</td><td><code>64 x 256</code> half</td><td>64 KiB</td><td>每槽等于 32 KiB</td></tr>
    <tr><td>L0C</td><td><code>240 x 256</code> float</td><td>240 KiB</td><td>小于 256 KiB</td></tr>
    <tr><td>Vector Core UB</td><td>输入和输出各 <code>120 x 256</code> float</td><td>240 KiB</td><td>小于 248 KiB 可用空间</td></tr>
  </tbody>
</table>

`Init` 绑定 GM 地址并分配 Vector Core UB，`Process` 声明各级静态 Tensor 并通过编译期断言检查容量。


In [ ]:
%%writefile -a Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv_operator.h

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::Init(
        __gm__ uint8_t* xMatrix, __gm__ uint8_t* yMatrix, __gm__ uint8_t* bias, __gm__ uint8_t* zMatrix)
    {
        aGMOri.SetGlobalBuffer((__gm__ half*)xMatrix);
        bGMOri.SetGlobalBuffer((__gm__ half*)yMatrix);
        biasGMOri.SetGlobalBuffer((__gm__ half*)bias);
        cGMOri.SetGlobalBuffer((__gm__ float*)zMatrix);

        // xUB 接收双目的 Fixpipe 输出；geluOutUB 保存 GELU 结果。
        xUB = AscendC::LocalTensor<float>(AscendC::TPosition::VECCALC, 0, baseM / 2 * baseN);
        geluOutUB = AscendC::LocalTensor<float>(AscendC::TPosition::VECCALC, geluOutUBAddr, baseM / 2 * baseN);
    }

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::Process()
    {
        InitComputeParams();
        static_assert(
            2 * baseM * baseK * stepKa * sizeof(half) + baseN * sizeof(half) <= L1_PINGPONG_BYTES,
            "A1 Ping/Pong and Bias exceed the first 256 KiB of L1");
        static_assert(
            2 * baseK * baseN * stepKb * sizeof(half) <= L1_PINGPONG_BYTES,
            "B1 Ping/Pong exceed the second 256 KiB of L1");
        static_assert(
            baseM * baseK * sizeof(half) <= L0_PINGPONG_BYTES,
            "A2 Ping/Pong buffer exceeds 32 KiB");
        static_assert(
            baseK * baseN * sizeof(half) <= L0_PINGPONG_BYTES,
            "B2 Ping/Pong buffer exceeds 32 KiB");
        static_assert(baseM * baseN * sizeof(float) <= 256 * 1024, "L0C tile exceeds 256 KiB");
        static_assert(
            baseM * baseN * sizeof(float) <= 248 * 1024,
            "Two Vector Core UB buffers together exceed the 3510 usable UB capacity");
        static_assert(
            singleCoreK % (baseK * stepKa) == 0 && singleCoreK % (baseK * stepKb) == 0,
            "singleCoreK must contain complete A/B L1 copy-in batches");
        static_assert(
            M % 2 == 0 && singleCoreM % 2 == 0 && baseM % 2 == 0,
            "M partitions must be even for dual-Vector-Core row splitting");

        // ============================================================
        // Cube Core 侧：Buffer 分配 + 计算循环 + Fixpipe 搬出
        // ============================================================
        uint32_t a1PingpongSize = baseM * baseK * stepKa;
        uint32_t b1PingpongSize = baseK * baseN * stepKb;
        uint32_t a2PingpongSize = baseM * baseK;
        uint32_t b2PingpongSize = baseK * baseN;

        AscendC::LocalTensor<half> a1LocalPing(AscendC::TPosition::A1, 0, a1PingpongSize);
        AscendC::LocalTensor<half> a1LocalPong(AscendC::TPosition::A1, a1PingpongSize * sizeof(half), a1PingpongSize);
        AscendC::LocalTensor<half> a2LocalPing(AscendC::TPosition::A2, 0, a2PingpongSize);
        AscendC::LocalTensor<half> a2LocalPong(AscendC::TPosition::A2, L0_PINGPONG_BYTES, a2PingpongSize);

        AscendC::LocalTensor<half> b1LocalPing(AscendC::TPosition::B1, L1_PINGPONG_BYTES, b1PingpongSize);
        AscendC::LocalTensor<half> b1LocalPong(
            AscendC::TPosition::B1, L1_PINGPONG_BYTES + b1PingpongSize * sizeof(half), b1PingpongSize);
        AscendC::LocalTensor<half> b2LocalPing(AscendC::TPosition::B2, 0, b2PingpongSize);
        AscendC::LocalTensor<half> b2LocalPong(AscendC::TPosition::B2, L0_PINGPONG_BYTES, b2PingpongSize);
        AscendC::LocalTensor<half> bias1Local(
            AscendC::TPosition::C1, 2 * a1PingpongSize * sizeof(half), baseN);
        AscendC::LocalTensor<float> bias2Local(AscendC::TPosition::C2, 0, baseN);
        AscendC::LocalTensor<float> cLocal(AscendC::TPosition::CO1, 0, baseM * baseN);

        if ASCEND_IS_AIC {
            InitAicSyncFlags();
        }

        for (uint32_t nBlockIdx = 0; nBlockIdx < nLoopCount; nBlockIdx++) {
            for (uint32_t mBlockIdx = 0; mBlockIdx < mLoopCount; mBlockIdx++) {
                if ASCEND_IS_AIC {
                    ProcessLoopAic(
                        a1LocalPing, a1LocalPong, a2LocalPing, a2LocalPong, b1LocalPing, b1LocalPong, b2LocalPing,
                        b2LocalPong, bias1Local, bias2Local, cLocal, mBlockIdx, nBlockIdx);
                }
                if ASCEND_IS_AIV {
                    ProcessLoopAiv(mBlockIdx, nBlockIdx);
                }
            }
        }

        if ASCEND_IS_AIC {
            WaitAicSyncFlags();
        }
    }


---
## 4. Cube Core 多级流水

Cube Core 采用两级 Double Buffer：MTE2 将 GM 数据批量搬入 L1，MTE1 将当前 K Tile 搬入 L0，Cube 同时计算上一 Tile。硬件事件保护 Ping/Pong 槽位的读写依赖。

### 4.1 单 Buffer 的依赖

![单 Buffer 与 Double Buffer 时序](images/07_02_static_tensor_cv_fusion_advanced/4_single_double_buffer.svg)

单 Buffer 复用同一 L1/L0 地址。MTE2 必须等待 MTE1 停止读取 L1，MTE1 必须等待 Cube 停止读取 L0，下一轮数据无法提前进入片上存储。解决这一依赖需要为 L1 和 L0 分别准备两个地址槽位。

### 4.2 L1 批量搬运与 L1/L0 数据路径

`stepKa=stepKb=4` 将四个连续 `baseK` 子块一次搬入 L1，MTE1 通过 `kOffsetInChunk` 选择当前 K Tile。批量搬运减少 MTE2 启动次数，并为下一批数据提前搬入提供时间窗口。

`DataCopyInA/B`、`DataLoadA/B`、`PrepareBias` 和 `Compute` 分别实现 GM-L1、L1-L0、Bias 准备与 MMAD。


In [ ]:
%%writefile -a Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv_operator.h

// GM → A1: 将 A 矩阵的 stepKa 个 baseM * baseK 子块批量搬入 L1
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::DataCopyInA(
    AscendC::LocalTensor<half> a1Local, uint32_t kChunkIdx, uint32_t mBlockIdx)
    {
        uint32_t curM = (mBlockIdx != baseMCount) ? baseM : tailM;
        AscendC::Nd2NzParams nd2nzParams;
        nd2nzParams.ndNum = 1;
        nd2nzParams.nValue = curM;
        nd2nzParams.dValue = baseK * stepKa;
        nd2nzParams.srcNdMatrixStride = 0;
        nd2nzParams.srcDValue = K;
        nd2nzParams.dstNzC0Stride = baseM;
        nd2nzParams.dstNzNStride = 1;
        nd2nzParams.dstNzMatrixStride = 0;
        AscendC::DataCopy(a1Local, aGM[kChunkIdx * baseK + mBlockIdx * K * baseM], nd2nzParams);
    }

// GM → B1: 将 B 矩阵的 stepKb 个 baseK * baseN 子块批量搬入 L1
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::DataCopyInB(
    AscendC::LocalTensor<half> b1Local, uint32_t kChunkIdx, uint32_t nBlockIdx)
    {
        uint32_t curN = (nBlockIdx != baseNCount) ? baseN : tailN;
        AscendC::Nd2NzParams nd2nzParams;
        if constexpr (!IS_B_TRANSPOSE) {
            nd2nzParams.ndNum = 1;
            nd2nzParams.nValue = baseK * stepKb;
            nd2nzParams.dValue = curN;
            nd2nzParams.srcNdMatrixStride = 0;
            nd2nzParams.srcDValue = N;
            nd2nzParams.dstNzC0Stride = baseK * stepKb;
            nd2nzParams.dstNzNStride = 1;
            nd2nzParams.dstNzMatrixStride = 0;
            AscendC::DataCopy(b1Local, bGM[kChunkIdx * baseK * N + nBlockIdx * baseN], nd2nzParams);
        } else {
            nd2nzParams.ndNum = 1;
            nd2nzParams.nValue = curN;
            nd2nzParams.dValue = baseK * stepKb;
            nd2nzParams.srcNdMatrixStride = 0;
            nd2nzParams.srcDValue = K;
            nd2nzParams.dstNzC0Stride = baseN;
            nd2nzParams.dstNzNStride = 1;
            nd2nzParams.dstNzMatrixStride = 0;
            AscendC::DataCopy(b1Local, bGM[kChunkIdx * baseK + nBlockIdx * baseN * K], nd2nzParams);
        }
    }

// A1 → A2: 将 L1 中的一个 baseM * baseK 搬入 L0
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::DataLoadA(
        AscendC::LocalTensor<half> a1Local, AscendC::LocalTensor<half> a2Local, uint32_t mBlockIdx,
        uint32_t kOffsetInChunkA)
    {
        uint32_t srcAddr = kOffsetInChunkA * baseK * baseM;
        uint32_t curMAlign = (mBlockIdx != baseMCount) ? baseM : tailMAlign;
#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 2201)
        AscendC::LoadData3DParamsV2<half> loadDataParams;
        loadDataParams.l1H = 1;
        loadDataParams.l1W = baseM;
        loadDataParams.channelSize = baseK;
        loadDataParams.kExtension = baseK;
        loadDataParams.mExtension = curMAlign;
        loadDataParams.mStartPt = 0;
        loadDataParams.kStartPt = 0;
        AscendC::LoadData(a2Local, a1Local[srcAddr], loadDataParams);
#elif defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        AscendC::LoadData2DParamsV2 loadDataParams;
        loadDataParams.mStartPosition = 0;
        loadDataParams.kStartPosition = 0;
        loadDataParams.mStep = DivCeil(curMAlign, CUBE_BLOCK);
        loadDataParams.kStep = DivCeil(baseK, CUBE_BLOCK);
        loadDataParams.srcStride = DivCeil(baseM, CUBE_BLOCK);
        loadDataParams.dstStride = DivCeil(curMAlign, CUBE_BLOCK);
        loadDataParams.sid = 0;
        loadDataParams.ifTranspose = false;
        AscendC::LoadData(a2Local, a1Local[srcAddr], loadDataParams);
#endif
    }

// B1 → B2: 将 L1 中的一个 baseK * baseN 搬入 L0
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::DataLoadB(
        AscendC::LocalTensor<half> b1Local, AscendC::LocalTensor<half> b2Local, uint32_t nBlockIdx,
        uint32_t kOffsetInChunkB)
    {
        uint32_t srcAddr = kOffsetInChunkB * baseK * (IS_B_TRANSPOSE ? baseN : CUBE_BLOCK);
        uint32_t curNAlign = (nBlockIdx != baseNCount) ? baseN : tailNAlign;
#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 2201)
        if constexpr (!IS_B_TRANSPOSE) {
            // B 非转置: 使用 LoadData（卷积数据搬运）v2 完成 [K, N] → [N, K] 转置搬运
            AscendC::LoadData3DParamsV2<half> loadDataParams;
            loadDataParams.l1H = 1;
            loadDataParams.l1W = baseK * stepKb;
            loadDataParams.channelSize = baseN;
            loadDataParams.kExtension = curNAlign;
            loadDataParams.mExtension = baseK;
            loadDataParams.mStartPt = kOffsetInChunkB * baseK;
            loadDataParams.kStartPt = 0;
            loadDataParams.strideW = 1;
            loadDataParams.strideH = 1;
            loadDataParams.filterW = 1;
            loadDataParams.filterH = 1;
            loadDataParams.dilationFilterW = 1;
            loadDataParams.dilationFilterH = 1;
            loadDataParams.filterSizeW = false;
            loadDataParams.filterSizeH = false;
            loadDataParams.enTranspose = true;
            loadDataParams.fMatrixCtrl = false;
            AscendC::LoadData(b2Local, b1Local, loadDataParams);
        } else {
            // B 转置: 按 CUBE_BLOCK 粒度逐块搬入 L0，无需转置
            AscendC::LoadData2DParams loadDataParams;
            uint32_t dstOffset = curNAlign * CUBE_BLOCK;
            uint32_t srcOffset = baseN * CUBE_BLOCK;
            loadDataParams.repeatTimes = DivCeil(curNAlign, CUBE_BLOCK);
            loadDataParams.srcStride = 1;
            loadDataParams.dstGap = 0;
            loadDataParams.ifTranspose = false;
            for (int i = 0; i < DivCeil(baseK, CUBE_BLOCK); ++i) {
                AscendC::LoadData(b2Local[i * dstOffset], b1Local[srcAddr + i * srcOffset], loadDataParams);
            }
        }
#elif defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        if constexpr (!IS_B_TRANSPOSE) {
            // B 非转置: 使用 LoadData2D V2 完成 [K, N] → [N, K] 转置搬运
            AscendC::LoadData2DParamsV2 loadDataParams;
            loadDataParams.mStartPosition = 0;
            loadDataParams.kStartPosition = 0;
            loadDataParams.mStep = DivCeil(baseK, CUBE_BLOCK);
            loadDataParams.kStep = DivCeil(curNAlign * sizeof(half), 32);
            loadDataParams.srcStride = DivCeil(baseK * stepKb, CUBE_BLOCK);
            loadDataParams.dstStride = DivCeil(curNAlign, CUBE_BLOCK);
            loadDataParams.ifTranspose = true;
            AscendC::LoadData(b2Local, b1Local[srcAddr], loadDataParams);
        } else {
            // B 转置: 无需转置，直接按块搬运
            AscendC::LoadData2DParamsV2 loadDataParams;
            loadDataParams.mStartPosition = 0;
            loadDataParams.kStartPosition = 0;
            loadDataParams.mStep = DivCeil(curNAlign, CUBE_BLOCK);
            loadDataParams.kStep = DivCeil(baseK * sizeof(half), 32);
            loadDataParams.srcStride = DivCeil(baseN, CUBE_BLOCK);
            loadDataParams.dstStride = DivCeil(curNAlign, CUBE_BLOCK);
            loadDataParams.ifTranspose = false;
            AscendC::LoadData(b2Local, b1Local[srcAddr], loadDataParams);
        }
#endif
    }

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::PrepareBias(
        AscendC::LocalTensor<half>& bias1Local, AscendC::LocalTensor<float>& bias2Local, uint32_t nBlockIdx)
    {
        AscendC::DataCopy(bias1Local, biasGM[nBlockIdx * baseN], baseN);
        AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);
        AscendC::DataCopy(
            bias2Local, bias1Local,
            {1, static_cast<uint16_t>(baseN * sizeof(float) / 64), 0, 0});
        AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(EVENT_ID2);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(EVENT_ID2);
    }

// 执行 MMAD 计算并累加到 CO1。
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::Compute(
        AscendC::LocalTensor<float> cLocal, AscendC::LocalTensor<half> a2Local, AscendC::LocalTensor<half> b2Local,
        AscendC::LocalTensor<float> bias2Local, uint32_t kBlockIdx, uint32_t mBlockIdx, uint32_t nBlockIdx)
    {
        AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(mte1DBFlag);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(mte1DBFlag);
        uint32_t curM = (mBlockIdx != baseMCount) ? baseM : tailM;
        uint32_t curN = (nBlockIdx != baseNCount) ? baseN : tailN;
        AscendC::MmadParams mmadParams;
        mmadParams.m = curM;
        mmadParams.n = curN;
        mmadParams.k = baseK;
        mmadParams.cmatrixInitVal = (kBlockIdx == 0);
        mmadParams.isBias = (kBlockIdx == 0);
        mmadParams.unitFlag = (kBlockIdx != kLoopCount - 1) ? 2 : 3;
        if (kBlockIdx == 0) {
            AscendC::Mmad(cLocal, a2Local, b2Local, bias2Local, mmadParams);
        } else {
            AscendC::Mmad(cLocal, a2Local, b2Local, mmadParams);
        }
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(mte1DBFlag);
        mte1DBFlag ^= 1;
    }


### 4.3 L1/L0 Double Buffer 与硬件事件

![L1/L0 Double Buffer](images/07_02_static_tensor_cv_fusion_advanced/4_double_buffer.svg)

L1 的 Ping/Pong 以四个 K Tile 为切换单位，L0 的 Ping/Pong 每个 K Tile 切换一次。两级缓冲区的切换周期不同，因此分别使用 L1 读写索引和 `mte1DBFlag`。

![K 方向多级流水](images/07_02_static_tensor_cv_fusion_advanced/4_multilevel_pipeline.svg)

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">事件方向</th><th align="left">语义</th><th align="left">保护对象</th></tr></thead>
  <tbody>
    <tr><td><code>MTE2_MTE1</code></td><td>L1 批量块就绪</td><td>MTE1 读取 A1/B1</td></tr>
    <tr><td><code>MTE1_MTE2</code></td><td>L1 槽位释放</td><td>MTE2 覆盖 A1/B1</td></tr>
    <tr><td><code>MTE1_M</code></td><td>L0 数据就绪</td><td>Cube 读取 A2/B2</td></tr>
    <tr><td><code>M_MTE1</code></td><td>L0 槽位释放</td><td>MTE1 覆盖 A2/B2</td></tr>
  </tbody>
</table>

K 循环根据 `kBlockIdx` 选择 L1/L0 Ping/Pong 槽位，并在当前 MMAD 期间预取下一批 L1 数据。


In [ ]:
%%writefile -a Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv_operator.h

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::
    InitAicSyncFlags()
    {
        // ============================================================
        // 同步标志初始化：预置反向事件，建立首次 WaitFlag 所需的初始可写状态
        // ============================================================
        AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID0);
        AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID1);
        AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID2);
        AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID3);
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(EVENT_ID0);
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(EVENT_ID1);
    }

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::
    WaitAicSyncFlags()
    {
        // ============================================================
        // 等待所有同步完成
        // ============================================================
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(EVENT_ID1);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID1);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID2);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID3);
    }

// Cube Core 侧单个 M/N 分块计算：DataCopyIn + DataLoad + Compute + Fixpipe 搬出
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::
    ProcessLoopAic(
        AscendC::LocalTensor<half>& a1LocalPing, AscendC::LocalTensor<half>& a1LocalPong,
        AscendC::LocalTensor<half>& a2LocalPing, AscendC::LocalTensor<half>& a2LocalPong,
        AscendC::LocalTensor<half>& b1LocalPing, AscendC::LocalTensor<half>& b1LocalPong,
        AscendC::LocalTensor<half>& b2LocalPing, AscendC::LocalTensor<half>& b2LocalPong,
        AscendC::LocalTensor<half>& bias1Local, AscendC::LocalTensor<float>& bias2Local,
        AscendC::LocalTensor<float>& cLocal, uint32_t mBlockIdx, uint32_t nBlockIdx)
    {
        // ============================================================
        // DataCopyIn 进度跟踪变量
        // ============================================================
        uint32_t a1NextKChunkIdx = 0;
        uint32_t b1NextKChunkIdx = 0;
        uint8_t a1CopyInIdx = 0;
        uint8_t b1CopyInIdx = 0;

        // Bias 地址由 N 分块确定。每个输出 tile 准备一次，K 循环复用 C2。
        PrepareBias(bias1Local, bias2Local, nBlockIdx);

        // ---- 搬入 A1 Ping 和 B1 Ping 的首个批量块 ----
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID0);
        DataCopyInA(a1LocalPing, a1NextKChunkIdx, mBlockIdx);
        AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);
        a1NextKChunkIdx += stepKa;
        a1CopyInIdx ^= 1;

        AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>(EVENT_ID2);
        DataCopyInB(b1LocalPing, b1NextKChunkIdx, nBlockIdx);
        AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID2);
        b1NextKChunkIdx += stepKb;
        b1CopyInIdx ^= 1;

        // ---- K 方向主循环 ----
        for (uint32_t kBlockIdx = 0; kBlockIdx < kLoopCount; kBlockIdx++) {
            uint32_t a1ReadIdx = (kBlockIdx / stepKa) % 2;
            uint32_t b1ReadIdx = (kBlockIdx / stepKb) % 2;
            uint32_t kOffsetInChunkA = kBlockIdx % stepKa;
            uint32_t kOffsetInChunkB = kBlockIdx % stepKb;

            AscendC::LocalTensor<half> a1ReadBuf = (a1ReadIdx == 0) ? a1LocalPing : a1LocalPong;
            AscendC::LocalTensor<half> b1ReadBuf = (b1ReadIdx == 0) ? b1LocalPing : b1LocalPong;

            AscendC::LocalTensor<half> a2Local = (mte1DBFlag == 0) ? a2LocalPing : a2LocalPong;
            AscendC::LocalTensor<half> b2Local = (mte1DBFlag == 0) ? b2LocalPing : b2LocalPong;

            // ---- 反向同步：等待上一轮 Compute 释放 L0 缓冲区 ----
            AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(mte1DBFlag);

            // ---- 正向同步：等待 L1 批量块数据就绪 ----
            if (kOffsetInChunkA == 0) {
                AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>((a1ReadIdx == 0) ? EVENT_ID0 : EVENT_ID1);
            }
            if (kOffsetInChunkB == 0) {
                AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>((b1ReadIdx == 0) ? EVENT_ID2 : EVENT_ID3);
            }

            // ---- DataLoad: L1 → L0 ----
            DataLoadA(a1ReadBuf, a2Local, mBlockIdx, kOffsetInChunkA);
            DataLoadB(b1ReadBuf, b2Local, nBlockIdx, kOffsetInChunkB);

            // ---- 反向同步：当前 L1 批量块消费完成，允许 DataCopyIn 覆盖对应缓冲区 ----
            if ((kOffsetInChunkA + 1) == stepKa) {
                AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>((a1ReadIdx == 0) ? EVENT_ID0 : EVENT_ID1);
            }
            if ((kOffsetInChunkB + 1) == stepKb) {
                AscendC::SetFlag<AscendC::HardEvent::MTE1_MTE2>((b1ReadIdx == 0) ? EVENT_ID2 : EVENT_ID3);
            }

            // ---- Compute: Mmad 矩阵乘累加 ----
            Compute(cLocal, a2Local, b2Local, bias2Local, kBlockIdx, mBlockIdx, nBlockIdx);

            // ---- 搬入下一个 L1 批量块，使 DataCopyIn 与 Compute 流水重叠 ----
            if (((kBlockIdx == 0) || ((kOffsetInChunkB + 1) == stepKb)) && b1NextKChunkIdx < kLoopCount) {
                AscendC::LocalTensor<half> b1WriteBuf = (b1CopyInIdx == 0) ? b1LocalPing : b1LocalPong;
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>((b1CopyInIdx == 0) ? EVENT_ID2 : EVENT_ID3);
                DataCopyInB(b1WriteBuf, b1NextKChunkIdx, nBlockIdx);
                AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>((b1CopyInIdx == 0) ? EVENT_ID2 : EVENT_ID3);
                b1NextKChunkIdx += stepKb;
                b1CopyInIdx ^= 1;
            }
            if (((kBlockIdx == 0) || ((kOffsetInChunkA + 1) == stepKa)) && a1NextKChunkIdx < kLoopCount) {
                AscendC::LocalTensor<half> a1WriteBuf = (a1CopyInIdx == 0) ? a1LocalPing : a1LocalPong;
                AscendC::WaitFlag<AscendC::HardEvent::MTE1_MTE2>((a1CopyInIdx == 0) ? EVENT_ID0 : EVENT_ID1);
                DataCopyInA(a1WriteBuf, a1NextKChunkIdx, mBlockIdx);
                AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>((a1CopyInIdx == 0) ? EVENT_ID0 : EVENT_ID1);
                a1NextKChunkIdx += stepKa;
                a1CopyInIdx ^= 1;
            }
        }

        // ---- CopyOut Cube Core 侧：Fixpipe L0C → 双 Vector Core UB ----
        CopyOutAic(cLocal, mBlockIdx, nBlockIdx);
    }


### 4.4 流水重叠判据

有效流水重叠需要同时满足四项条件：Ping/Pong 地址不重叠；写入方与读取方分别运行在不同的硬件流水线上；下一批 CopyIn 在当前 MMAD 完成前发起；正向和反向事件分别表示数据就绪与槽位释放。`PipeUtilization` 中 MTE2、MTE1 与 Cube 的活跃区间用于验证重叠是否成立。


---
## 5. Vector Core RegBase

Cube Core 生成输出 Tile 后，Vector Core 对数据执行 GELU。RegBase 将连续算术链的中间值保存在寄存器中，减少 UB 读写和 `PIPE_V` 屏障。

### 5.1 GELU 算术链

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">序号</th><th align="left">指令</th><th align="left">结果</th></tr></thead>
  <tbody>
    <tr><td>1</td><td><code>Mul</code></td><td><code>x^2</code></td></tr>
    <tr><td>2</td><td><code>Mul</code></td><td><code>x^3</code></td></tr>
    <tr><td>3</td><td><code>Muls</code></td><td><code>0.044715 x^3</code></td></tr>
    <tr><td>4</td><td><code>Add</code></td><td><code>x + 0.044715 x^3</code></td></tr>
    <tr><td>5</td><td><code>Muls</code></td><td><code>-1.595769(...)</code></td></tr>
    <tr><td>6</td><td><code>Exp</code></td><td>分母指数项</td></tr>
    <tr><td>7</td><td><code>Adds</code></td><td>完整分母</td></tr>
    <tr><td>8</td><td><code>Div</code></td><td>GELU 输出</td></tr>
  </tbody>
</table>

### 5.2 MemBase 与 RegBase 数据流

![GELU 的 MemBase 与 RegBase 数据流](images/07_02_static_tensor_cv_fusion_advanced/5_regbase_comparison.svg)

MemBase 需要把每一级中间结果写回 UB，并在连续依赖之间插入 `PIPE_V` 屏障。RegBase 每个 Vector Length 只执行一次 `LoadAlign` 和一次 `StoreAlign`，八级算术链的中间值保存在 `RegTensor` 中。

### 5.3 RegBase GELU 实现

`GeluVf` 每轮处理一个 Vector Length，循环步长为 `oneRepeatSize`。


In [ ]:
%%writefile Source/07_02_static_tensor_cv_fusion_advanced/gelu_unroll_practice.h
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#ifndef ASCENDC_07_02_GELU_UNROLL_PRACTICE_H
#define ASCENDC_07_02_GELU_UNROLL_PRACTICE_H

// 单路 RegBase GELU。课后实践在此基础上完成双路展开。
__simd_vf__ inline void GeluVf(__ubuf__ float* xAddr, __ubuf__ float* yAddr, uint32_t n)
{
    constexpr uint32_t oneRepeatSize = AscendC::GetVecLen() / sizeof(float);
    uint32_t loopNum = DivCeil(n, oneRepeatSize);
    AscendC::Reg::MaskReg mask;
    AscendC::Reg::RegTensor<float> xReg, yReg;

    for (uint16_t i = 0; i < loopNum; ++i) {
        mask = AscendC::Reg::UpdateMask<float>(n);
        AscendC::Reg::LoadAlign(xReg, xAddr + i * oneRepeatSize);
        AscendC::Reg::Mul(yReg, xReg, xReg, mask);
        AscendC::Reg::Mul(yReg, yReg, xReg, mask);
        AscendC::Reg::Muls(yReg, yReg, GELU_COEFF_A, mask);
        AscendC::Reg::Add(yReg, xReg, yReg, mask);
        AscendC::Reg::Muls(yReg, yReg, GELU_COEFF_B, mask);
        AscendC::Reg::Exp(yReg, yReg, mask);
        AscendC::Reg::Adds(yReg, yReg, 1.0f, mask);
        AscendC::Reg::Div(yReg, xReg, yReg, mask);
        AscendC::Reg::StoreAlign(yAddr + i * oneRepeatSize, yReg, mask);
    }
}

#endif // ASCENDC_07_02_GELU_UNROLL_PRACTICE_H


### 5.4 接入 Vector Core 处理路径

`ProcessLoopAiv` 在每个 M/N Tile 上调用 `GeluRegBaseCompute`。`GeluRegBaseCompute` 从静态 Tensor 取得 UB 物理地址，再调用 `GeluVf`。


In [ ]:
%%writefile -a Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv_operator.h

// Vector Core 侧单个 M/N 分块计算
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::
    ProcessLoopAiv(uint32_t mBlockIdx, uint32_t nBlockIdx)
    {
        GeluFromUB(mBlockIdx, nBlockIdx);
    }

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::
    GeluRegBaseCompute(
        const AscendC::LocalTensor<float>& xLocal, const AscendC::LocalTensor<float>& yLocal, uint32_t n)
    {
        __ubuf__ float* xAddr = (__ubuf__ float*)xLocal.GetPhyAddr();
        __ubuf__ float* yAddr = (__ubuf__ float*)yLocal.GetPhyAddr();
        GeluVf(xAddr, yAddr, n);
    }


### 5.5 尾块 Mask

`oneRepeatSize = GetVecLen() / sizeof(float)`。`UpdateMask<float>(n)` 每轮递减剩余元素数，并为最后一轮生成有效位掩码。Load、八条算术指令和 Store 使用同一个 Mask，避免尾块越界读写。


---
## 6. CV 数据交接

Fixpipe 将 L0C 结果沿 M 维分发到两个 Vector Core 的 UB。跨核事件保证 Vector Core 在 Fixpipe 完成后读取数据，GELU 结果通过 MTE3 写回 GM。

### 6.1 Fixpipe 双目标分区

![L0C-UB 双目标数据路径](images/07_02_static_tensor_cv_fusion_advanced/6_cv_data_path.svg)

`dualDstCtl=0b01` 沿 M 维拆分 Fixpipe 输出。`baseM=240` 时，两个 Vector Core 分别接收 120 行，每个 Vector Core 的输入为 `120 x 256` 个 `float`。

### 6.2 跨核同步与结果写回

Cube Core 在 Fixpipe 完成后发送事件，两个 Vector Core 等待同一事件，执行 RegBase GELU，再通过 `GetSubBlockIdx()` 计算各自的 GM 输出偏移。

`CopyOutAic` 配置 Fixpipe 双目标输出，`GeluFromUB` 完成跨核等待、GELU 计算和 GM 写回；`mmad_gelu_adv` 是 Host 侧启动的 Kernel 入口。


In [ ]:
%%writefile -a Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv_operator.h

// Cube Core 侧：Fixpipe 将 L0C 按 M 维均分到两个 Vector Core 的 UB。
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::CopyOutAic(
    AscendC::LocalTensor<float> cLocal, uint32_t mBlockIdx, uint32_t nBlockIdx)
    {
        uint32_t curMAlign = (mBlockIdx != baseMCount) ? baseM : tailMAlign;
        uint32_t curM = (mBlockIdx != baseMCount) ? baseM : tailM;
        uint32_t curN = (nBlockIdx != baseNCount) ? baseN : tailN;

        AscendC::FixpipeParamsArch3510<AscendC::CO2Layout::ROW_MAJOR> fixpipeParams;
        fixpipeParams.mSize = DivCeil(curM, 2) * 2;
        fixpipeParams.nSize = curN;
        fixpipeParams.srcStride = curMAlign;
        fixpipeParams.dstStride = curN;
        fixpipeParams.dualDstCtl = 0b01;
        fixpipeParams.unitFlag = 3;
        AscendC::Fixpipe<float, float, CFG_ROW_MAJOR_UB>(xUB, cLocal, fixpipeParams);

        AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(0x8);
    }

// Vector Core 侧：L0C-UB 直通、RegBase GELU、UB-GM 搬出。
template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__aicore__ inline void
KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb>::
    GeluFromUB(uint32_t mBlockIdx, uint32_t nBlockIdx)
    {
        AscendC::CrossCoreWaitFlag(0x8);

        uint32_t curM = (mBlockIdx != baseMCount) ? baseM : tailM;
        uint32_t curN = (nBlockIdx != baseNCount) ? baseN : tailN;

        // dualDstCtl=0b01 按 M 维拆分，每个 Vector Core 持有 M/2 行。
        uint32_t halfM = curM / 2;
        uint32_t computeLen = halfM * curN;

        GeluRegBaseCompute(xUB, geluOutUB, computeLen);

        AscendC::SetFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);

        uint32_t localSubIdx = AscendC::GetSubBlockIdx() % 2;
        uint32_t offset = halfM * N;
        AscendC::DataCopyParams copyParams;
        copyParams.blockCount = halfM;
        copyParams.blockLen = curN * sizeof(float);
        copyParams.srcStride = 0;
        copyParams.dstStride = (N - curN) * sizeof(float);
        AscendC::DataCopyPad<float>(
            cGM[mBlockIdx * baseM * N + nBlockIdx * baseN + localSubIdx * offset], geluOutUB, copyParams);
    }

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t baseM, uint32_t baseK, uint32_t baseN, uint32_t singleCoreM,
    uint32_t singleCoreK, uint32_t singleCoreN, uint32_t stepKa, uint32_t stepKb>
__global__ __mix__(1, 2) void mmad_gelu_adv(
    __gm__ uint8_t* xMatrix, __gm__ uint8_t* yMatrix, __gm__ uint8_t* bias, __gm__ uint8_t* zMatrix)
{
    AscendC::InitSocState();
    KernelMmadGelu<M, K, N, baseM, baseK, baseN, singleCoreM, singleCoreK, singleCoreN, stepKa, stepKb> op;
    op.Init(xMatrix, yMatrix, bias, zMatrix);
    op.Process();
    AscendC::PipeBarrier<PIPE_ALL>();
}

#endif  // MMAD_GELU_ADV_OPERATOR_H


In [ ]:
%%bash
operator_path="Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv_operator.h"
host_path="Source/07_02_static_tensor_cv_fusion_advanced/mmad_gelu_adv.asc"

test -s "${operator_path}"
test -s "${host_path}"
grep -q '#endif  // MMAD_GELU_ADV_OPERATOR_H' "${operator_path}"
grep -q '#include "mmad_gelu_adv_operator.h"' "${host_path}"
echo "算子实现已补全: ${operator_path}"
echo "Host 源文件: ${host_path}"


---
## 7. 编译、精度与性能

### 7.1 编译运行

`run.sh` 构建并执行唯一目标 `mmad_gelu_adv`，随后完成输入生成与全量精度校验。


In [ ]:
%%bash
cd Source/07_02_static_tensor_cv_fusion_advanced
bash run.sh


### 7.2 精度标准

Golden 使用 float32 MMAD 结果和 float64 GELU 公式计算，指数输入裁剪到 `[-88, 88]` 后转为 float32。校验条件为 `rtol=1e-3`、`atol=1e-3`，不一致元素比例不超过 `1e-3`。指数裁剪用于抑制 NumPy 参考计算溢出，不影响硬件关注区间内的 GELU 结果。


### 7.3 性能采集方法

性能验证分为两组实验：第 7.4 节衡量多级流水与 RegBase 组合后的算子总体收益，第 7.5 节固定 Cube Core 路径，仅比较 Vector 阶段的 MemBase 与 RegBase 实现。两组实验使用不同指标，不能将总体 `Task Duration` 与局部 `aiv_vec_time` 直接比较。

#### 7.3.1 对照边界

同 Shape 总体对照固定输入规模、Tile、核心映射、输出类型和 L0C-UB 交接，只改变待评估的计算实现。其目的不是归因某一个优化点，而是衡量高级实现相对串行实现的组合收益。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">对照项</th><th align="left">同 Shape 对照</th><th align="left"><code>mmad_gelu_adv</code></th></tr></thead>
  <tbody>
    <tr><td>Shape 与输出类型</td><td><code>M=1920, K=2048, N=2048</code>，float32 输出</td><td>相同</td></tr>
    <tr><td>Tile 与核心映射</td><td><code>240 x 64 x 256</code>，<code>numBlocks=32</code></td><td>相同</td></tr>
    <tr><td>CV 数据交接</td><td>L0C-UB，两个 Vector Core 按 M 维分区</td><td>相同</td></tr>
    <tr><td>Cube Core</td><td>串行 K 循环</td><td>L1/L0 Double Buffer 多级流水</td></tr>
    <tr><td>Vector Core</td><td>MemBase GELU，显式关闭 SIMD VF 融合</td><td>RegBase GELU</td></tr>
  </tbody>
</table>

#### 7.3.2 采集流程

两种实现均在同一台 3510 上运行，设备频率固定为 1650 MHz。每种实现先完成 10 次预热，再使用 `msprof op` 独立采集 5 轮；每轮重新启动采集任务，避免把同一轮中的重复执行当作独立样本。最后取 5 轮 `Task Duration` 的中位数作为总体性能结果。

Profiler 之外再执行 100 次稳态计时，用于确认总体趋势。稳态计时包含 Host 调度开销，因此只作为交叉验证，不替代设备侧 `Task Duration`。具体运行命令见第 7.6 节。

#### 7.3.3 指标口径

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">指标</th><th align="left">用途</th><th align="left">对应章节</th></tr></thead>
  <tbody>
    <tr><td><code>Task Duration</code></td><td>衡量完整融合 Kernel 的设备侧执行时间</td><td>7.4 总体性能</td></tr>
    <tr><td>AIC Time、MTE2/MTE1/Cube、Cube Ratio</td><td>判断 Cube Core 各流水的活跃时间和重叠程度</td><td>7.4 总体性能</td></tr>
    <tr><td><code>aiv_vec_time</code></td><td>隔离 GELU 的 Vector 计算时间</td><td>7.5 RegBase 对照</td></tr>
    <tr><td>Host 稳态均值</td><td>在无 Profiler 条件下交叉验证总体趋势</td><td>7.4、7.6</td></tr>
  </tbody>
</table>


### 7.4 同 Shape 总体性能

![MMAD-GELU 同 Shape 性能对照](images/07_02_static_tensor_cv_fusion_advanced/7_baseline_vs_advanced.svg)

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">指标</th><th align="left">同 Shape 对照</th><th align="left"><code>mmad_gelu_adv</code></th><th align="left">说明</th></tr></thead>
  <tbody>
    <tr><td>Task Duration 中位数</td><td>77.360 us</td><td>46.465 us</td><td>降低 39.94%，加速 1.665 倍</td></tr>
    <tr><td>5 轮 Task Duration</td><td>77.285 / 76.828 / 77.841 / 77.708 / 77.360 us</td><td>46.575 / 46.465 / 46.309 / 46.691 / 46.277 us</td><td><code>msprof op</code> 独立采样</td></tr>
    <tr><td>AIC Time</td><td>70.627 us</td><td>41.536 us</td><td>中位轮次，Block 0</td></tr>
    <tr><td>AIC MTE2 / MTE1 / Cube</td><td>57.601 / 10.716 / 37.236 us</td><td>32.759 / 12.335 / 37.236 us</td><td>各流水活跃时间允许重叠</td></tr>
    <tr><td>Cube Ratio</td><td>52.72%</td><td>89.65%</td><td>中位轮次，Block 0</td></tr>
    <tr><td>AIV Time / Vector Time</td><td>76.154 / 7.447 us</td><td>45.258 / 3.499 us</td><td>Block 0 / Vector 0</td></tr>
    <tr><td>有效 MMAD 吞吐</td><td>208.20 TFLOPS</td><td>346.63 TFLOPS</td><td><code>2MKN / Task Duration</code></td></tr>
    <tr><td>Host 稳态均值</td><td>60.674 us</td><td>45.450 us</td><td>10 次预热，100 次执行</td></tr>
    <tr><td>精度</td><td>error ratio = 0</td><td>error ratio = 0</td><td>相同 Golden，全量校验</td></tr>
  </tbody>
</table>

Cube 活跃时间保持 37.236 us，说明矩阵计算量不变；Cube Ratio 从 52.72% 提高到 89.65%，说明多级流水缩短了等待区间。各流水活跃时间之和大于 AIC Time，表明 MTE2、MTE1 与 Cube 存在重叠。


### 7.5 RegBase 与 MemBase Vector 性能对照

![GELU MemBase 与 RegBase 性能对比](images/07_02_static_tensor_cv_fusion_advanced/7_regbase_performance.svg)

纯 GELU 隔离实验固定公式、Shape、64 个 Vector Core、`tileLen=8192` 和 GM-UB 路径，只切换 MemBase 与 RegBase。Vector 计算性能使用五轮 `msprof` 的 Block 0 `aiv_vec_time` 中位数。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">指标</th><th align="left">MemBase</th><th align="left">RegBase</th><th align="left">变化</th></tr></thead>
  <tbody>
    <tr><td>5 轮 Block 0 <code>aiv_vec_time</code></td><td>142.688 / 142.652 / 142.670 / 142.688 / 142.688 us</td><td>64.689 / 64.689 / 64.686 / 64.689 / 64.671 us</td><td>独立采样</td></tr>
    <tr><td>Block 0 <code>aiv_vec_time</code> 中位数</td><td>142.688 us</td><td>64.689 us</td><td>降低 54.66%，加速 2.206 倍</td></tr>
    <tr><td>显式 <code>PIPE_V</code> 屏障</td><td>7</td><td>0</td><td>依赖由寄存器指令链表达</td></tr>
    <tr><td>中间结果位置</td><td>UB</td><td><code>RegTensor</code></td><td>消除中间 UB 往返</td></tr>
  </tbody>
</table>

纯 GELU 场景中，RegBase 将 Block 0 `aiv_vec_time` 从 142.688 us 降至 64.689 us，降低 54.66%。中间结果保存在 `RegTensor` 中，消除了七次显式 `PIPE_V` 屏障和中间结果的 UB 往返。

融合算子实验保持 Cube Core 流水、Shape、Tile 和 CV 交接不变，仅切换 GELU 实现。该对比仍采用 Block 0 `aiv_vec_time`，用于隔离 Vector 阶段的实现差异：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead><tr><th align="left">指标</th><th align="left">MemBase</th><th align="left">RegBase</th><th align="left">变化</th></tr></thead>
  <tbody>
    <tr><td>Block 0 <code>aiv_vec_time</code></td><td>7.447 us</td><td>3.499 us</td><td>降低 53.01%，加速 2.128 倍</td></tr>
    <tr><td>精度</td><td>error ratio = 0</td><td>error ratio = 0</td><td>相同 Golden</td></tr>
  </tbody>
</table>

两个场景的 `aiv_vec_time` 分别降低 54.66% 和 53.01%，说明 RegBase 对该 GELU 算术链的 Vector 计算耗时具有稳定收益。该对比不包含 Cube Core 流水或其他算子阶段。

> **提示**：对于连续的 Vector 计算链，RegBase 可减少中间结果的 UB 往返和同步开销，通常具有性能优势；MemBase 的适用范围和兼容性更好。实际开发时，应结合目标硬件、计算链特征和兼容性要求选择实现方式。


### 7.6 稳态运行与 Profiler 采集

稳态计时包含 10 次预热和 100 次执行。随后使用 `msprof op` 采集一次 `PipeUtilization`，结果保存在 `prof_mmad_gelu_adv` 目录。


In [ ]:
%%bash
cd Source/07_02_static_tensor_cv_fusion_advanced
CV_WARMUP_COUNT=10 CV_BENCHMARK_COUNT=100 ./build/mmad_gelu_adv

rm -rf prof_mmad_gelu_adv
CV_WARMUP_COUNT=0 CV_BENCHMARK_COUNT=1 msprof op \
  --application=./build/mmad_gelu_adv \
  --output=./prof_mmad_gelu_adv \
  --aic-metrics=PipeUtilization \
  --launch-count=1 \
  --warm-up=0


---
## 8. 小结

`mmad_gelu_adv` 使用 32 个 AI Core 完成 M/N 二维分区。Cube Core 通过 L1/L0 Double Buffer 重叠 MTE2、MTE1 与 MMAD，两个 Vector Core 通过 L0C-UB 路径接收结果并执行 RegBase GELU。

同 Shape 测量中，Task Duration 从 77.360 us 降至 46.465 us。该数值包含多级流水、RegBase 和 CV 数据交接的组合收益；RegBase 的局部影响由第 7.5 节受控实验给出。


---
## 课后实践

请基于本节 `mmad_gelu_adv` 融合样例，完成 GELU 寄存器双路展开。

目标：保持算子名称、Shape、Tile、Cube Core 流水、CV 交接和数学语义不变，将 `GeluVf` 改为每轮处理两个 Vector Length。

<table style="border-collapse: collapse; width: auto; margin: 12px auto 18px 0; border: 1px solid #ddd;">
  <thead><tr><th align="left">项目</th><th align="left">要求</th></tr></thead>
  <tbody>
    <tr><td>寄存器</td><td>使用两组 <code>xReg/yReg</code> 与两组 Mask</td></tr>
    <tr><td>循环步长</td><td>每轮处理两个 Vector Length</td></tr>
    <tr><td>尾块</td><td>主循环处理完整双路，奇数尾块在循环后单独处理</td></tr>
    <tr><td>数据流</td><td>每个 Vector Length 保持一次 LoadAlign 和一次 StoreAlign</td></tr>
    <tr><td>精度</td><td>通过原始 Golden 的全量校验</td></tr>
  </tbody>
</table>

请直接编辑下方 AscendC 代码单元；完成后执行紧随其后的验证单元。


In [ ]:
%%writefile Source/07_02_static_tensor_cv_fusion_advanced/gelu_unroll_practice.h
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS SOFTWARE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#ifndef ASCENDC_07_02_GELU_UNROLL_PRACTICE_H
#define ASCENDC_07_02_GELU_UNROLL_PRACTICE_H

// TODO: 将单路 RegBase GELU 改为两路展开。
__simd_vf__ inline void GeluVf(__ubuf__ float* xAddr, __ubuf__ float* yAddr, uint32_t n)
{
    constexpr uint32_t oneRepeatSize = AscendC::GetVecLen() / sizeof(float);
    uint32_t loopNum = DivCeil(n, oneRepeatSize);
    AscendC::Reg::MaskReg mask;
    AscendC::Reg::RegTensor<float> xReg, yReg;

    for (uint16_t i = 0; i < loopNum; ++i) {
        mask = AscendC::Reg::UpdateMask<float>(n);
        AscendC::Reg::LoadAlign(xReg, xAddr + i * oneRepeatSize);
        AscendC::Reg::Mul(yReg, xReg, xReg, mask);
        AscendC::Reg::Mul(yReg, yReg, xReg, mask);
        AscendC::Reg::Muls(yReg, yReg, GELU_COEFF_A, mask);
        AscendC::Reg::Add(yReg, xReg, yReg, mask);
        AscendC::Reg::Muls(yReg, yReg, GELU_COEFF_B, mask);
        AscendC::Reg::Exp(yReg, yReg, mask);
        AscendC::Reg::Adds(yReg, yReg, 1.0f, mask);
        AscendC::Reg::Div(yReg, xReg, yReg, mask);
        AscendC::Reg::StoreAlign(yAddr + i * oneRepeatSize, yReg, mask);
    }
}

#endif // ASCENDC_07_02_GELU_UNROLL_PRACTICE_H


### 实践验证

完成修改后，执行 `run.sh` 重新构建并验证结果。预期输出包含 `error ratio: 0.0000` 和 `test pass!`。


In [ ]:
!cd Source/07_02_static_tensor_cv_fusion_advanced && bash run.sh


### 参考答案

参考答案如下。


In [ ]:
!cat answer/07_02_static_tensor_cv_fusion_advanced/gelu_unroll_practice.h
